In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from IPython.display import display, Audio, Markdown
from openai import OpenAI
from dotenv import load_dotenv
import io
import tempfile
import re
import os

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
os.environ["OPENAI_BASE_URL"] = os.getenv('BASE_URL')
client = OpenAI()

class AgentState(TypedDict):
    input_text: str
    processed_text: str
    audio_data: bytes
    audio_path: str
    content_type: str
def classify_content(state: AgentState) -> AgentState:
    """将输入文本分类为以下四种类型之一：普通 (general)、诗歌 (poem)、新闻 (news) 或笑话 (joke)。"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "请将内容分类为以下类别之一：'general' (普通), 'poem' (诗歌), 'news' (新闻), 'joke' (笑话)。只需返回类别关键词。"},
            {"role": "user", "content": state["input_text"]}
        ]
    )
    state["content_type"] = response.choices[0].message.content.strip().lower()
    return state

def process_general(state: AgentState) -> AgentState:
    """处理普通内容（不进行特殊处理，直接返回原文本）。"""
    state["processed_text"] = state["input_text"]
    return state

def process_poem(state: AgentState) -> AgentState:
    """将输入文本作为诗歌处理，将其重写为优美的诗歌风格。"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "请将以下文本重写为一段简短而优美的诗句："},
            {"role": "user", "content": state["input_text"]}
        ]
    )
    state["processed_text"] = response.choices[0].message.content.strip()
    return state

def process_news(state: AgentState) -> AgentState:
    """将输入文本作为新闻处理，将其重写为正式的新闻播报风格。"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "请以正式的新闻主播口吻重写以下文本："},
            {"role": "user", "content": state["input_text"]}
        ]
    )
    state["processed_text"] = response.choices[0].message.content.strip()
    return state

def process_joke(state: AgentState) -> AgentState:
    """将输入文本作为笑话处理，将其转化为一个简短有趣的幽默段子。"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "请将以下文本转化成一个简短有趣的笑话："},
            {"role": "user", "content": state["input_text"]}
        ]
    )
    state["processed_text"] = response.choices[0].message.content.strip()
    return state

def text_to_speech(state: AgentState, save_file: bool = False) -> AgentState:
    """
    根据内容类型匹配相应的音色，将处理后的文本转换为语音。
    可选：将音频保存到文件。

    参数:
        state (AgentState): 包含处理后的文本和内容类型的状态字典。
        save_file (bool, 可选): 如果为 True，则保存音频到文件。默认为 False。

    返回:
        AgentState: 更新后的状态，包含音频数据和文件路径（如果已保存）。
    """
    
    # 将内容类型映射到特定音色，默认为 "alloy"
    # alloy: 中性, nova: 温柔(适合诗歌), onyx: 沉稳(适合新闻), shimmer: 活泼(适合笑话)
    voice_map = {
        "general": "alloy",
        "poem": "nova",
        "news": "onyx",
        "joke": "shimmer"
    }
    voice = voice_map.get(state["content_type"], "alloy")
    
    audio_data = io.BytesIO()

    # 生成语音并将音频流写入内存
    with client.audio.speech.with_streaming_response.create(
        model="tts-1",
        voice=voice,
        input=state["processed_text"]
    ) as response:
        for chunk in response.iter_bytes():
            audio_data.write(chunk)
    
    state["audio_data"] = audio_data.getvalue()
    
    # 如果有需求，将音频保存到临时文件
    if save_file:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as temp_audio:
            temp_audio.write(state["audio_data"])
            state["audio_path"] = temp_audio.name
    else:
        state["audio_path"] = ""
    
    return state
# 定义图工作流
workflow = StateGraph(AgentState)

# 向图中添加节点 (Nodes)
# 每个节点对应之前定义的处理函数
workflow.add_node("classify_content", classify_content)  # 分类节点
workflow.add_node("process_general", process_general)    # 普通处理节点
workflow.add_node("process_poem", process_poem)          # 诗歌处理节点
workflow.add_node("process_news", process_news)          # 新闻处理节点
workflow.add_node("process_joke", process_joke)          # 笑话处理节点
workflow.add_node("text_to_speech", text_to_speech)      # 语音合成节点

# 设置图的入口点
# 整个流程将从“分类内容”开始
workflow.set_entry_point("classify_content")

# 定义基于内容类型的条件边 (Conditional Edges)
# 根据 classify_content 返回的 content_type 决定下一步走哪个分支
workflow.add_conditional_edges(
    "classify_content",
    lambda x: x["content_type"],
    {
        "general": "process_general",  # 如果是普通内容，跳转到 process_general
        "poem": "process_poem",        # 如果是诗歌，跳转到 process_poem
        "news": "process_news",        # 如果是新闻，跳转到 process_news
        "joke": "process_joke",        # 如果是笑话，跳转到 process_joke
    }
)

# 将所有的处理器节点连接到语音合成节点 (Edges)
# 无论走哪个分支，最后都会执行文本转语音
workflow.add_edge("process_general", "text_to_speech")
workflow.add_edge("process_poem", "text_to_speech")
workflow.add_edge("process_news", "text_to_speech")
workflow.add_edge("process_joke", "text_to_speech")

# 设置结束点（通常语音合成是最后一步）
workflow.add_edge("text_to_speech", END)

# 编译工作流
app = workflow.compile()
def sanitize_filename(text, max_length=20):
    """将文本转换为有效且简洁的文件名。"""
    # 移除所有非字母数字、非空格和非连字符的字符
    sanitized = re.sub(r'[^\w\s-]', '', text.lower())
    # 将空格和连字符统一替换为下划线
    sanitized = re.sub(r'[-\s]+', '_', sanitized)
    return sanitized[:max_length]

def run_tts_agent_and_play(input_text: str, content_type: str, save_file: bool = True):
    """运行语音合成智能体并播放音频。"""
    # 调用编译好的 LangGraph 应用
    result = app.invoke({
        "input_text": input_text, 
        "processed_text": "", 
        "audio_data": b"",
        "audio_path": "", 
        "content_type": content_type
    })
    
    print(f"检测到的内容类型: {result['content_type']}")
    print(f"处理后的文本内容: {result['processed_text']}")
    
    # 播放音频（仅在本地 Jupyter 环境中有效）
    display(Audio(result['audio_data'], autoplay=True))
    
    if save_file:
        # 在笔记本上一级目录创建 'audio' 文件夹
        audio_dir = os.path.join('..', 'audio')
        os.makedirs(audio_dir, exist_ok=True)
        
        # 清洗文件名
        sanitized_text = sanitize_filename(input_text)
        file_name = f"{content_type}_{sanitized_text}.mp3"
        file_path = os.path.join(audio_dir, file_name)
        
        # 写入音频文件
        with open(file_path, "wb") as f:
            f.write(result['audio_data'])
        
        print(f"音频已保存至: {file_path}")
        
        # 为 GitHub 生成相对路径下载链接
        github_relative_path = f"../audio/{file_name}"
        display(Markdown(f"[下载 {content_type} 音频: {sanitized_text}]({github_relative_path})"))
        
        # 关于 GitHub 预览的提示
        print("提示：GitHub 不支持直接播放音频，请使用下载链接收听。")
    else:
        print("未执行音频保存操作。")
    
    return result

# 准备测试示例
examples = {
    "general": "我每天吃五顿饭。",
    "poem": "玫瑰是红色的，紫罗兰是蓝色的，糖果是甜的，你也是最棒的。",
    "news": "突发新闻：科学家在马里亚纳海沟发现了一种新型深海生物。",
    "joke": "为什么程序员喜欢在黑暗中工作？因为他们不喜欢光（light）！"
}

# 遍历并处理所有示例
for content_type, text in examples.items():
    print(f"\n正在处理 {content_type} 类别的示例：")
    print(f"输入文本: {text}")
    
    # 运行 TTS 智能体并保存文件
    result = run_tts_agent_and_play(text, content_type, save_file=True)
    
    print("-" * 50)

print("所有示例处理完毕。您可以使用上方的链接下载音频文件。")


正在处理 general 类别的示例：
输入文本: 我每天吃五顿饭。
检测到的内容类型: general
处理后的文本内容: 我每天吃五顿饭。


音频已保存至: ../audio/general_我每天吃五顿饭.mp3


[下载 general 音频: 我每天吃五顿饭](../audio/general_我每天吃五顿饭.mp3)

提示：GitHub 不支持直接播放音频，请使用下载链接收听。
--------------------------------------------------

正在处理 poem 类别的示例：
输入文本: 玫瑰是红色的，紫罗兰是蓝色的，糖果是甜的，你也是最棒的。
检测到的内容类型: poem
处理后的文本内容: 数据汇聚至今秋，知识蓄势待发求。


音频已保存至: ../audio/poem_玫瑰是红色的紫罗兰是蓝色的糖果是甜的你也.mp3


...

[下载 news 音频: 突发新闻科学家在马里亚纳海沟发现了一种新](../audio/news_突发新闻科学家在马里亚纳海沟发现了一种新.mp3)

提示：GitHub 不支持直接播放音频，请使用下载链接收听。
--------------------------------------------------

正在处理 joke 类别的示例：
输入文本: 为什么程序员喜欢在黑暗中工作？因为他们不喜欢光（light）！
检测到的内容类型: joke
处理后的文本内容: 为什么程序员喜欢在黑暗中工作？因为他们只想找代码中的“bug”，而不是“虫子”！


音频已保存至: ../audio/joke_为什么程序员喜欢在黑暗中工作因为他们不喜.mp3


[下载 joke 音频: 为什么程序员喜欢在黑暗中工作因为他们不喜](../audio/joke_为什么程序员喜欢在黑暗中工作因为他们不喜.mp3)

提示：GitHub 不支持直接播放音频，请使用下载链接收听。
--------------------------------------------------
所有示例处理完毕。您可以使用上方的链接下载音频文件。
